# Building Blocks with `nn.Sequential`

Stack layers and track **output shapes** through the pipeline.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. Define a Sequential model

In [ ]:
class ShapeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(8, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, 10),
        )
    def forward(self, x):
        return self.features(x)

model = ShapeNet()
x = torch.randn(4, 8)
print(model)

## 2. Hook intermediate shapes

In [ ]:
shapes = [('input', tuple(x.shape))]
h = x
for i, layer in enumerate(model.features):
    h = layer(h)
    shapes.append((f'{i}: {layer.__class__.__name__}', tuple(h.shape)))
for name, sh in shapes:
    print(f"{name:25s} → {sh}")

## 3. Diagram: layer blocks with shapes

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
labels = [s[0] for s in shapes]
ypos = np.arange(len(labels))
ax.barh(ypos, [int(np.prod(s[1])) if len(s[1])>1 else s[1][0] for s in shapes],
        color=plt.cm.viridis(np.linspace(0.2, 0.9, len(labels))))
ax.set_yticks(ypos); ax.set_yticklabels(labels)
ax.set_xlabel('tensor elements (batch flattened)')
ax.set_title('nn.Sequential — output size per stage')
plt.tight_layout(); plt.show()

## 4. Activation map after first hidden layer

In [ ]:
with torch.no_grad():
    h1 = model.features[1](model.features[0](x))
fig, ax = plt.subplots(figsize=(8, 4))
ax.imshow(h1.numpy(), aspect='auto', cmap='magma')
ax.set_title('Hidden activations after Linear+ReLU'); ax.set_xlabel('neuron')
plt.tight_layout(); plt.show()

## 5. Parameter count per block

In [ ]:
blocks = ['Linear(8→16)', 'ReLU', 'Linear(16→32)', 'ReLU', 'Linear(32→10)']
params = [8*16+16, 0, 16*32+32, 0, 32*10+10]
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(blocks, params, color='teal', edgecolor='black')
ax.set_title('Learnable parameters per layer'); ax.set_ylabel('# params')
plt.xticks(rotation=15); plt.tight_layout(); plt.show()